# Lesson 5: Set power before turning the key ON

**Keep the physical key in STANDBY/OFF throughout this notebook.**
Read the state, store one approved power setpoint, then verify it.
Only the power setting changes. For an independent run with the key already ON, see [lesson 6](06_key_on_and_run.ipynb).

`?K` is the physical key position; `?L` is the reported laser state.
With `?K = 0`, shutter closed and active faults clear, preparation accepts
`?L = 0` or `2`. State 2 means fault-off; preserve its history and review it
before key ON. It is not permission to enable emission.

Run cells individually with one serial owner. **Stop after an error**; inspect
`laser.last_transaction`, disconnect and resolve the cause before starting again.

## 1. Settings

Confirm the connection settings, approved target and power ceiling.
The examples compare the no-active-fault `?F` payload to `b"SYSTEM OK"`.

With `prompt_enabled=True`, the controller reads the reply through CR/LF, then
consumes the enabled trailing prompt before sending another instruction. The
instrument prompt must already be enabled; construction sends no echo or prompt
changes. Matching instruction echoes and recognized leading notices are removed
from `read()` payloads while their received bytes remain in transaction evidence.

`TIMEOUT_S` configures serial write and receive waits. A reply line, each leading
notice and the trailing prompt use separate `read_until()` calls with that
configured timeout; there is no combined transaction deadline. No extra pause
is inserted between queries. Restart the kernel after a package update.

Saved outputs and execution counts are retained from earlier hardware runs;
they have not been regenerated for these revised cells.

In [1]:
PORT = "COM5"
MODEL = "V2"  # V2, V5 or V6
BAUDRATE = 19200
TIMEOUT_S = 2.0  # Serial write and receive timeout
TARGET_W = 0.01  # Operator-approved watts, in 0.01 W increments
POWER_LIMIT_W = 0.1  # Site-approved ceiling in watts

## 2. Create the controller

Construction sends nothing. Require 0.01 W increments and check the transmitted
value against the ceiling before opening the port.

In [2]:
from math import ulp

from coherent_verdi import VerdiController

if PORT == "<PORT>" or POWER_LIMIT_W is None:
    raise ValueError("Fill in every setting in section 1 first.")

laser = VerdiController(
    PORT,
    model=MODEL,
    baudrate=BAUDRATE,
    timeout_s=TIMEOUT_S,
    prompt_enabled=True,
    allow_writes=True,
    power_limit_w=POWER_LIMIT_W,
)
if type(TARGET_W) not in (int, float) or not 0 <= TARGET_W <= laser.power_limit_w:
    raise ValueError("TARGET_W must be a finite number between 0 and POWER_LIMIT_W.")
stepped_target = round(TARGET_W, 2)
if abs(TARGET_W - stepped_target) > 2 * ulp(TARGET_W):
    raise ValueError("TARGET_W must be in 0.01 W increments (e.g. 0.01, 0.02, 0.03).")
target_bytes = b"%.4f" % stepped_target
if float(target_bytes) > laser.power_limit_w:
    raise ValueError("Rounded TARGET_W exceeds POWER_LIMIT_W.")
print("Will send:", repr(b"P=" + target_bytes))

Will send: b'P=0.0100'


## 3. Open the serial port

Close other serial owners first. Run this cell and inspect the result.
`True` means the local port opened; no instrument command has been sent.

In [3]:
connected = laser.connect(override=True)
print("Serial port open:", connected)

Serial port open: True


## 4. Read the starting state with the key OFF

Clear host buffers once, then read the version, state, stored power and fault history.
`read()` returns payload `bytes` with echo, CR/LF framing, recognized leading
firmware notices and prompt removed. It does not convert the payload to numbers,
booleans or fault codes. Bytes returned by serial reads, including notices and incomplete replies returned
at a timeout, remain in `laser.last_transaction["response"]`.

Saved outputs and execution counts below are retained from the earlier hardware run;
they have not been regenerated after this startup-order correction.

In [4]:
laser.purge_buffers()

In [5]:
laser.read(b"?SV")

b'6.05'

In [6]:
print("Version:", repr(laser.read(b"?SV")))

before = {q: laser.read(q) for q in (b"?K", b"?L", b"?S", b"?F", b"?SP", b"?FH")}
for query, reply in before.items():
    print(query, repr(reply))
if before[b"?K"] != b"0":
    raise RuntimeError("Require physical key OFF; key ON comes last.")
if before[b"?L"] not in (b"0", b"2") or before[b"?S"] != b"0" or before[b"?F"] != b"SYSTEM OK":
    raise RuntimeError("Require laser OFF, shutter closed and no active faults.")

Version: b'6.05'
b'?K' b'0'
b'?L' b'0'
b'?S' b'0'
b'?F' b'SYSTEM OK'
b'?SP' b'0.0100'
b'?FH' b'SYSTEM OK'


## 5. Store the power once

Type `SET`, then recheck the key and state before writing.
`set_power_w()` sends only `P=...` and validates the command acknowledgment.
It does not enable the laser or move the shutter.

In [7]:
if input("With the physical key OFF, type SET to store power: ").strip() != "SET":
    raise RuntimeError("Cancelled. No settings were changed.")

now = {q: laser.read(q) for q in (b"?K", b"?L", b"?S", b"?F")}
if now[b"?K"] != b"0":
    raise RuntimeError("Require physical key OFF; key ON comes last.")
if now[b"?L"] not in (b"0", b"2") or now[b"?S"] != b"0" or now[b"?F"] != b"SYSTEM OK":
    raise RuntimeError("Require laser OFF, shutter closed and no active faults.")

laser.set_power_w(TARGET_W)
print("Acknowledgment:", repr(laser.last_transaction["response"]))

Acknowledgment: b'\r\nVerdi V-2>'


## 6. Verify the stored setpoint

`?SP` is the stored setpoint, not measured optical power.
Confirm the requested value, key still OFF and unchanged laser state.

In [8]:
after = {q: laser.read(q) for q in (b"?SP", b"?K", b"?L", b"?S", b"?F")}
for query, reply in after.items():
    print(query, repr(reply))
if after[b"?SP"] != target_bytes:
    raise RuntimeError("Setpoint readback mismatch.")
if after[b"?K"] != b"0":
    raise RuntimeError("Require physical key OFF; key ON comes last.")
if after[b"?L"] != before[b"?L"] or after[b"?S"] != b"0" or after[b"?F"] != b"SYSTEM OK":
    raise RuntimeError("State or faults changed after the write.")
print("Stored power verified; keep the physical key OFF until ready to run.")

b'?SP' b'0.0100'
b'?K' b'0'
b'?L' b'0'
b'?S' b'0'
b'?F' b'SYSTEM OK'
Stored power verified; keep the physical key OFF until ready to run.


## 7. Disconnect

Release the serial port. The setpoint remains stored; disconnecting changes no
laser setting. If a cell failed, inspect `laser.last_transaction` before disconnecting.

In [9]:
print(laser.last_transaction)

{'instruction': b'?F', 'request': b'?F\r\n', 'response': b'SYSTEM OK\r\nVerdi V-2>', 'payload': b'SYSTEM OK', 'outcome': 'received', 'error': None}


In [10]:
laser.disconnect()